# MMM Analysis: ECOSTRESS vs CRW Thermal Comparison — Belize 2023

Compares ECOSTRESS land-surface temperature (70 m) against NOAA CoralWatch (CRW) SST (5 km) for Belizean MPAs during the 2023 bleaching event. Anomalies are computed relative to the site-specific CRW Maximum Monthly Mean (MMM).

## Data requirements
Place the following files in a `data/` directory before running:
- `ECOSTRESS_LST_Mean_Cloudmasked_Composite_May-Nov23.tif` — cloud-masked ECOSTRESS composite
- `dhw_5km.nc` — NOAA CRW 5 km SST product
- `ct5km_climatology_v3.1.nc` — CRW climatology (MMM source)
- `mean_bleaching_surveys_per_location_Jun-Dec2023.csv` — bleaching survey data
- `Belize_MPAs_Reprojected.shp` (+ sidecar files) — Belize MPA shapefile


In [ ]:
# Install additional dependencies (only needed once)
# !pip install cartopy rioxarray


## 1. Per-MPA MMM Comparison Table
Produces a styled table of CRW MMM, ECOSTRESS LST, CRW SST, and MPA-specific MMM anomalies per MPA.


In [ ]:
# ================================================================
#  MPA MMM COMPARISON TABLE
#  Per MPA: CRW MMM, ECOSTRESS LST, CRW SST, MMM-relative anomalies
# ================================================================

import numpy as np
import pandas as pd
import xarray as xr
import rioxarray
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from tqdm import tqdm

# ── PATHS ─────────────────────────────────────────────────────────────────────
# Update these paths to point to your local copies of each file.
ECO_PATH      = "data/ECOSTRESS_LST_Mean_Cloudmasked_Composite_May-Nov23.tif"
CRW_PATH      = "data/dhw_5km.nc"
CLIM_PATH     = "data/ct5km_climatology_v3.1.nc"
BLEACHING_CSV = "data/mean_bleaching_surveys_per_location_Jun-Dec2023.csv"
MPA_SHP       = "data/Belize_MPAs_Reprojected.shp"

LAT_COL    = 'obs_lat'
LON_COL    = 'obs_lon'
TARGET_CRS = "EPSG:4269"
LAT_MIN, LAT_MAX = 15.7, 18.5
LON_MIN, LON_MAX = -89.5, -87.3

# ── 1. LOAD ───────────────────────────────────────────────────────────────────
print("Loading data...")
sites = pd.read_csv(BLEACHING_CSV).dropna(subset=[LAT_COL, LON_COL])
mpas  = gpd.read_file(MPA_SHP).to_crs(epsg=4326)[["NAME","geometry"]]

# ── 2. BUILD CRW MMM ──────────────────────────────────────────────────────────
print("Building CRW MMM...")
ds_clim = xr.open_dataset(CLIM_PATH)
use_360 = float(ds_clim.lon.max()) > 180
lon_min_sel = LON_MIN + 360 if use_360 else LON_MIN
lon_max_sel = LON_MAX + 360 if use_360 else LON_MAX
months = ["january","february","march","april","may","june",
          "july","august","september","october","november","december"]
layers = [ds_clim[f"sst_clim_{m}"].sel(lat=slice(LAT_MAX, LAT_MIN),
          lon=slice(lon_min_sel, lon_max_sel)) for m in months]
mmm = xr.concat(layers, dim=pd.Index(range(1, 13), name="month")).max(dim="month")
lons_mmm = mmm.lon.values
if use_360:
    lons_mmm = np.where(lons_mmm > 180, lons_mmm - 360, lons_mmm)
mmm_plot = mmm.assign_coords(lon=("lon", lons_mmm))

# ── HELPER: sample a raster at each survey site (5x5 pixel window) ────────────
def sample_raster(raster, df, desc=""):
    """Return a list of mean pixel values sampled around each survey site."""
    nodata = raster.rio.nodata
    rx, ry = len(raster.x), len(raster.y)
    results = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        try:
            nx = raster.sel(x=row[LON_COL], method='nearest').x.item()
            ny = raster.sel(y=row[LAT_COL], method='nearest').y.item()
            cx = raster.indexes['x'].get_loc(nx)
            cy = raster.indexes['y'].get_loc(ny)
            window = raster.isel(
                x=slice(max(0, cx-2), min(rx-1, cx+2)+1),
                y=slice(max(0, cy-2), min(ry-1, cy+2)+1)
            )
            if nodata is not None:
                window = window.where(window != nodata)
            results.append(window.where(window > 0).mean().item())
        except Exception:
            results.append(np.nan)
    return results

# ── 3. SAMPLE ECOSTRESS LST ───────────────────────────────────────────────────
print("Sampling ECOSTRESS LST...")
lst_raster = rioxarray.open_rasterio(ECO_PATH)
if lst_raster.rio.crs.to_string() != TARGET_CRS:
    lst_raster = lst_raster.rio.reproject(TARGET_CRS)
if 'band' in lst_raster.dims:
    lst_raster = lst_raster.squeeze('band', drop=True)

df = sites.copy()
df['LST_Value'] = sample_raster(lst_raster, df, desc="ECO LST")
df['MMM_CRW']   = [
    round(mmm_plot.sel(lat=row[LAT_COL], lon=row[LON_COL], method="nearest").item(), 4)
    for _, row in df.iterrows()
]

# ── 4. SAMPLE CRW SST ─────────────────────────────────────────────────────────
print("Sampling CRW SST...")
crw_ds = xr.open_dataset(CRW_PATH)
crw_ds = crw_ds.rio.set_spatial_dims(x_dim='longitude', y_dim='latitude')
crw_ds = crw_ds.rio.write_crs("EPSG:4326")
if crw_ds.rio.crs.to_string() != TARGET_CRS:
    crw_ds = crw_ds.rio.reproject(TARGET_CRS)
sst_raster = crw_ds['CRW_SST'].sel(
    time=slice("2023-05-01", "2023-11-30")).mean(dim='time', skipna=True)

df['SST_Value'] = sample_raster(sst_raster, df, desc="CRW SST")

# ── 5. COMPUTE ANOMALIES ──────────────────────────────────────────────────────
df['ECO_Anomaly_vs_MMM']   = df['LST_Value'] - df['MMM_CRW']
df['CRW_Anomaly_vs_MMM']   = df['SST_Value'] - df['MMM_CRW']

# ── 6. SPATIAL JOIN TO MPAs ───────────────────────────────────────────────────
print("Joining to MPAs...")
gdf = gpd.GeoDataFrame(df,
      geometry=gpd.points_from_xy(df[LON_COL], df[LAT_COL]),
      crs="EPSG:4326")
joined = gpd.sjoin(gdf, mpas, how="left", predicate="within")
joined = joined[joined["NAME"].notna()]

# ── 7. AGGREGATE PER MPA ──────────────────────────────────────────────────────
table = joined.groupby("NAME").agg(
    n_sites              = ("LST_Value",            "count"),
    Mean_MMM_CRW         = ("MMM_CRW",              "mean"),
    Mean_ECO_LST         = ("LST_Value",             "mean"),
    Mean_CRW_SST         = ("SST_Value",             "mean"),
    ECO_Anomaly_vs_MMM   = ("ECO_Anomaly_vs_MMM",   "mean"),
    CRW_Anomaly_vs_MMM   = ("CRW_Anomaly_vs_MMM",   "mean"),).reset_index()

for col in table.columns[1:]:
    table[col] = table[col].round(3)

table = table.sort_values('ECO_Anomaly_vs_MMM', ascending=False).reset_index(drop=True)
table.columns = [
    "MPA Name", "n",
    "CRW MMM (°C)",
    "ECOSTRESS LST (°C)",
    "CRW SST (°C)",
    "ECO Anom.\nvs MMM (°C)",
    "CRW Anom.\nvs MMM (°C)",]

print("\n── MPA Comparison Table ──────────────────────────────")
print(table.to_string(index=False))

# ── 8. PLOT TABLE ─────────────────────────────────────────────────────────────
print("\nPlotting table...")
n_rows = len(table)
n_cols = len(table.columns)
fig, ax = plt.subplots(figsize=(20, n_rows * 0.65 + 1.2))
ax.set_position([0.01, 0.02, 0.98, 0.88])
ax.axis("off")

tbl = ax.table(
    cellText  = table.values,
    colLabels = table.columns,
    cellLoc   = "center",
    loc       = "center",
    bbox      = [0, 0, 1, 1]
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.auto_set_column_width(col=list(range(n_cols)))
tbl.scale(1, 1.6)

# Header styling
for j in range(len(table.columns)):
    tbl[(0, j)].set_facecolor("#2c7bb6")
    tbl[(0, j)].set_text_props(color="white", fontweight="bold")

# Heatmap helper
def make_norm(vals):
    d_min, d_max = vals.min(), vals.max()
    if d_min < 0 < d_max:
        return mcolors.TwoSlopeNorm(vmin=d_min, vcenter=0, vmax=d_max), plt.cm.RdBu_r
    return mcolors.Normalize(vmin=d_min, vmax=d_max), plt.cm.YlOrRd

eco_anom_vals = table["ECO Anom.\nvs MMM (°C)"].values
crw_anom_vals = table["CRW Anom.\nvs MMM (°C)"].values
eco_norm, eco_cmap = make_norm(eco_anom_vals)
crw_norm, crw_cmap = make_norm(crw_anom_vals)

# Column indices: ECO anom vs MMM = 5, CRW anom vs MMM = 6  (0-based after 'MPA Name','n','CRW MMM','ECO LST','CRW SST')
heatmap_cols = {
    5: (eco_anom_vals, eco_norm, eco_cmap),
    6: (crw_anom_vals, crw_norm, crw_cmap),
}

for i in range(1, len(table) + 1):
    base = "#f9f9f9" if i % 2 == 0 else "white"
    for j in range(len(table.columns)):
        tbl[(i, j)].set_facecolor(base)
    for col_idx, (vals, norm, cmap) in heatmap_cols.items():
        rgba = cmap(norm(vals[i-1]))
        light = tuple(0.35 + 0.65*c if idx < 3 else c for idx, c in enumerate(rgba))
        tbl[(i, col_idx)].set_facecolor(light)

ax.set_title(
    "Per-MPA Thermal Comparison — Belize 2023\n"
    "ECOSTRESS (70 m) vs CRW (5 km)  |  MPA-specific MMM",
    fontsize=11, fontweight='bold', pad=15
)

plt.tight_layout()
plt.savefig("figures/mpa_mmm_comparison_table.png", dpi=300, bbox_inches='tight')
plt.show()
print("Done.")


## 2. MPA Heatwave Severity Maps & Bar Charts
Side-by-side choropleth maps and bar charts showing mean thermal anomaly per MPA, with bleaching percentage overlaid.


In [ ]:
# ================================================================
#  MPA HEATWAVE SEVERITY
#  Bar charts: mean thermal anomaly per MPA + mean bleaching % overlay
# ================================================================

import numpy as np
import pandas as pd
import xarray as xr
import rioxarray
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.gridspec import GridSpec
from tqdm import tqdm

# ── PATHS ─────────────────────────────────────────────────────────────────────
# (Same files as Cell 1 — update if running this cell independently.)
ECO_PATH      = "data/ECOSTRESS_LST_Mean_Cloudmasked_Composite_May-Nov23.tif"
CRW_PATH      = "data/dhw_5km.nc"
CLIM_PATH     = "data/ct5km_climatology_v3.1.nc"
BLEACHING_CSV = "data/mean_bleaching_surveys_per_location_Jun-Dec2023.csv"
MPA_SHP       = "data/Belize_MPAs_Reprojected.shp"

LAT_COL    = 'obs_lat'
LON_COL    = 'obs_lon'
BLEACH_COL = 'Bleached_Percent'
LAT_MIN, LAT_MAX = 15.85, 18.4
LON_MIN, LON_MAX = -88.8, -87.3
TARGET_CRS = "EPSG:4269"
EXTENT = [LON_MIN - 0.05, LON_MAX + 0.05, LAT_MIN - 0.05, LAT_MAX + 0.05]

# ── LOAD ──────────────────────────────────────────────────────────────────────
print("Loading...")
sites     = pd.read_csv(BLEACHING_CSV).dropna(subset=[LAT_COL, LON_COL])
mpas_4326 = gpd.read_file(MPA_SHP).to_crs(epsg=4326)[["NAME", "geometry"]].copy()

# Build CRW MMM
ds_clim = xr.open_dataset(CLIM_PATH)
use_360 = float(ds_clim.lon.max()) > 180
lon_min_sel = LON_MIN + 360 if use_360 else LON_MIN
lon_max_sel = LON_MAX + 360 if use_360 else LON_MAX
months  = ["january","february","march","april","may","june",
           "july","august","september","october","november","december"]
layers  = [ds_clim[f"sst_clim_{m}"].sel(lat=slice(LAT_MAX, LAT_MIN),
           lon=slice(lon_min_sel, lon_max_sel)) for m in months]
mmm     = xr.concat(layers, dim=pd.Index(range(1, 13), name="month")).max(dim="month")
lons_mmm = mmm.lon.values
if use_360:
    lons_mmm = np.where(lons_mmm > 180, lons_mmm - 360, lons_mmm)
mmm_plot = mmm.assign_coords(lon=("lon", lons_mmm))

def get_site_mmm(df):
    return [round(mmm_plot.sel(lat=r[LAT_COL], lon=r[LON_COL],
                               method="nearest").item(), 4)
            for _, r in df.iterrows()]

def sample_raster(raster, df, desc=""):
    """Return mean pixel values (5x5 window) for each survey site."""
    nodata = raster.rio.nodata
    rx, ry = len(raster.x), len(raster.y)
    results = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        try:
            nx = raster.sel(x=row[LON_COL], method='nearest').x.item()
            ny = raster.sel(y=row[LAT_COL], method='nearest').y.item()
            cx = raster.indexes['x'].get_loc(nx)
            cy = raster.indexes['y'].get_loc(ny)
            window = raster.isel(
                x=slice(max(0, cx-2), min(rx-1, cx+2)+1),
                y=slice(max(0, cy-2), min(ry-1, cy+2)+1)
            )
            if nodata is not None:
                window = window.where(window != nodata)
            results.append(window.where(window > 0).mean().item())
        except Exception:
            results.append(np.nan)
    return results

# ECOSTRESS
lst_raster = rioxarray.open_rasterio(ECO_PATH)
if lst_raster.rio.crs.to_string() != TARGET_CRS:
    lst_raster = lst_raster.rio.reproject(TARGET_CRS)
if 'band' in lst_raster.dims:
    lst_raster = lst_raster.squeeze('band', drop=True)

df_eco = sites.copy()
df_eco['LST_Value'] = sample_raster(lst_raster, df_eco, desc="ECO LST")
df_eco['MMM']       = get_site_mmm(df_eco)
df_eco['Anomaly']   = df_eco['LST_Value'] - df_eco['MMM']
df_eco.dropna(subset=['Anomaly'], inplace=True)

# CRW SST
crw_ds = xr.open_dataset(CRW_PATH)
crw_ds = crw_ds.rio.set_spatial_dims(x_dim='longitude', y_dim='latitude')
crw_ds = crw_ds.rio.write_crs("EPSG:4326")
if crw_ds.rio.crs.to_string() != TARGET_CRS:
    crw_ds = crw_ds.rio.reproject(TARGET_CRS)
sst_raster = crw_ds['CRW_SST'].sel(
    time=slice("2023-05-01", "2023-11-30")).mean(dim='time', skipna=True)

df_crw = sites.copy()
df_crw['SST_Value'] = sample_raster(sst_raster, df_crw, desc="CRW SST")
df_crw['MMM']       = get_site_mmm(df_crw)
df_crw['Anomaly']   = df_crw['SST_Value'] - df_crw['MMM']
df_crw.dropna(subset=['Anomaly'], inplace=True)

# MPA aggregation
def get_mpa_stats(df_sites, anom_col):
    gdf = gpd.GeoDataFrame(df_sites,
          geometry=gpd.points_from_xy(df_sites[LON_COL], df_sites[LAT_COL]),
          crs="EPSG:4326")
    joined = gpd.sjoin(gdf, mpas_4326, how="left", predicate="within")
    joined = joined[joined["NAME"].notna()]
    agg = joined.groupby("NAME").agg(
        mean_anomaly = (anom_col,   "mean"),
        std_anomaly  = (anom_col,   "std"),
        n_sites      = (anom_col,   "count"),
        mean_bleach  = (BLEACH_COL, "mean"),
        std_bleach   = (BLEACH_COL, "std"),
    ).reset_index()
    agg['std_anomaly'] = agg['std_anomaly'].fillna(0)
    agg['std_bleach']  = agg['std_bleach'].fillna(0)
    return agg

eco_mpa  = get_mpa_stats(df_eco, 'Anomaly')
crw_mpa  = get_mpa_stats(df_crw, 'Anomaly')
mpas_eco = mpas_4326.merge(eco_mpa, on="NAME", how="left")
mpas_crw = mpas_4326.merge(crw_mpa, on="NAME", how="left")

# Shared color scale
all_anom = pd.concat([eco_mpa['mean_anomaly'], crw_mpa['mean_anomaly']]).dropna()
VMIN = max(0.5, all_anom.min() - 0.05)
VMAX = all_anom.max() + 0.05
cmap = plt.cm.YlOrRd
norm = mcolors.Normalize(vmin=VMIN, vmax=VMAX)

def shorten(name):
    return (name
        .replace('High Protection for Biodiversity', 'High Prot.')
        .replace('Marine Reserve', 'MR')
        .replace('Wildlife Sanctuary', 'WS')
        .replace('Natural Monument', 'NM')
        .replace('National Park', 'NP')
        .replace('Spawning Aggregation Site Reserves', 'Spawning Res.')
        .replace('Turneffe Atolls', 'Turneffe')
    )

# ── FIGURE ────────────────────────────────────────────────────────────────────
print("Plotting...")
fig = plt.figure(figsize=(15, 14))
gs  = GridSpec(2, 2, figure=fig,
               width_ratios=[1.4, 1.0],
               hspace=0.15, wspace=0.18,
               left=0.03, right=0.97,
               top=0.93,  bottom=0.04)

def draw_map(pos, mpas_gdf, sites_df):
    ax = fig.add_subplot(pos, projection=ccrs.PlateCarree())
    ax.set_extent(EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND,      facecolor="#e8e0d0", zorder=2)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.7, zorder=4)
    ax.add_feature(cfeature.BORDERS,   linewidth=0.4, linestyle="--", zorder=4)
    gl = ax.gridlines(draw_labels=True, linewidth=0.3,
                      color="grey", alpha=0.4, linestyle="--")
    gl.top_labels = False; gl.right_labels = False
    gl.xlabel_style = {'size': 7}; gl.ylabel_style = {'size': 7}

    mpas_gdf[mpas_gdf['mean_anomaly'].isna()].plot(
        ax=ax, facecolor="#cccccc", edgecolor="white",
        linewidth=0.8, transform=ccrs.PlateCarree(), zorder=3)
    mpas_gdf[mpas_gdf['mean_anomaly'].notna()].plot(
        ax=ax, column='mean_anomaly', cmap=cmap, norm=norm,
        edgecolor="white", linewidth=0.8,
        transform=ccrs.PlateCarree(), zorder=3)
    ax.scatter(sites_df[LON_COL], sites_df[LAT_COL],
               color="black", s=25, edgecolors="white",
               linewidths=0.4, transform=ccrs.PlateCarree(), zorder=5)

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, orientation="vertical",
                        pad=0.02, shrink=0.7, aspect=20)
    cbar.set_label("Mean Anomaly (°C)", fontsize=8)
    cbar.ax.tick_params(labelsize=7)
    return ax

def draw_bar(pos, mpa_stats, xlabel):
    ax = fig.add_subplot(pos)
    df_plot = mpa_stats.sort_values('mean_anomaly', ascending=True).copy()
    df_plot['short_name'] = df_plot['NAME'].apply(shorten)
    colors_bar = [cmap(norm(v)) for v in df_plot['mean_anomaly']]

    ax.barh(range(len(df_plot)), df_plot['mean_anomaly'],
            color=colors_bar, edgecolor='grey', linewidth=0.4, height=0.65)
    ax.set_yticks(range(len(df_plot)))
    ax.set_yticklabels(
        [f"{row['short_name']}\n(n={int(row['n_sites'])})"
         for _, row in df_plot.iterrows()],
        fontsize=8.5, linespacing=1.4
    )

    # Bleaching % overlay on secondary x-axis
    ax2 = ax.twiny()
    ax2.set_xlim(0, 100)
    ax2.scatter(df_plot['mean_bleach'], range(len(df_plot)),
                color='#2166ac', s=40, zorder=5,
                edgecolors='white', linewidths=0.5)
    ax2.errorbar(df_plot['mean_bleach'], range(len(df_plot)),
                 xerr=df_plot['std_bleach'], fmt='none',
                 ecolor='#2166ac', capsize=3,
                 capthick=0.8, elinewidth=0.8, alpha=0.6)
    ax2.set_xlabel("Mean Bleaching (%)", fontsize=9, color='#2166ac')
    ax2.tick_params(axis='x', labelsize=8, colors='#2166ac')
    ax2.spines['top'].set_color('#2166ac')

    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_xlim(VMIN - 0.05, VMAX + 0.35)
    ax.tick_params(axis='x', labelsize=8)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.yaxis.set_tick_params(length=0)
    return ax

# Row 1 — ECOSTRESS
ax_a = draw_map(gs[0, 0], mpas_eco, df_eco)
ax_b = draw_bar(gs[0, 1], eco_mpa, xlabel="Mean LST Anomaly (°C)")
ax_a.text(-0.18, 0.5, "ECOSTRESS (70m)", transform=ax_a.transAxes,
          fontsize=9, fontweight="bold", va="center", ha="center", rotation=90)

# Row 2 — CRW
draw_map(gs[1, 0], mpas_crw, df_crw)
draw_bar(gs[1, 1], crw_mpa, xlabel="Mean SST Anomaly (°C)")
ax_a.text(-0.18, -0.5, "CRW (5km)", transform=ax_a.transAxes,
          fontsize=9, fontweight="bold", va="center", ha="center", rotation=90)

# Panel labels
for ax_panel, lbl in [(ax_a, "A"), (ax_b, "B")]:
    ax_panel.annotate(lbl, xy=(0.01, 0.99), xycoords='axes fraction',
                      fontsize=11, fontweight='bold', va='top', ha='left')

# Legend
site_patch = mpatches.Patch(facecolor="black", edgecolor="white",
                             label=f"Survey sites (n={len(sites)})")
grey_patch  = mpatches.Patch(facecolor="#cccccc", edgecolor="white",
                             label="MPA — no survey sites")
ax_a.legend(handles=[site_patch, grey_patch],
            loc="upper left", fontsize=7.5,
            framealpha=0.85, edgecolor="grey")

plt.savefig("figures/mpa_heatwave_severity.png", dpi=300, bbox_inches='tight')
plt.show()
print("Done.")


## 3. MPA Summary Statistics
Prints per-MPA anomaly and bleaching statistics, plus a within-atoll breakdown for Turneffe.


In [ ]:
# ── PRINT MPA STATS ───────────────────────────────────────────────────────────
# Requires eco_mpa, crw_mpa, df_eco from the previous cell.

cols = ['NAME', 'mean_anomaly', 'std_anomaly', 'n_sites', 'mean_bleach', 'std_bleach']

print("\n=== ECOSTRESS MPA Stats (sorted by mean anomaly) ===")
print(eco_mpa.sort_values('mean_anomaly', ascending=False)[cols].to_string(index=False))

print("\n=== CRW MPA Stats (sorted by mean anomaly) ===")
print(crw_mpa.sort_values('mean_anomaly', ascending=False)[cols].to_string(index=False))

# Within-atoll breakdown for Turneffe
print("\n=== Turneffe sites (ECOSTRESS) ===")
turneffe_mask = mpas_4326[mpas_4326['NAME'].str.contains('Turneffe', case=False)].geometry
turneffe_sites = df_eco[df_eco.apply(
    lambda r: turneffe_mask.contains(
        gpd.points_from_xy([r[LON_COL]], [r[LAT_COL]])[0]
    ).any(), axis=1
)]
print(turneffe_sites[[LAT_COL, LON_COL, BLEACH_COL, 'Anomaly']])
